# ChefEar STT Whisper Large-v3-Turbo QLoRA 실험 기록

## 1. 하이퍼파라미터 튜닝

> 학습 데이터: train300  
> 평가 기준: fixed100 / test_audio_100  
> 목적: LoRA 및 학습 설정 최적화

| 실험 | 변경값 | WER | CER | 판단 |
|---|---|---:|---:|---|
| 기준 2ep | r=16, alpha=32, LR=1e-4, q/v, dropout=0.05 | 33.20% | 5.81% | 기준 |
| 실험1 | LR 1e-4 → 5e-5 | 40.46% | 7.51% | 악화 |
| 실험2 | r=16 → 32 | 34.13% | 5.99% | 소폭 악화 |
| 실험3 | alpha=32 → 64 | 28.22% | 4.85% | 개선 |
| 실험4 | target=q,k,v,out | **20.12%** | **3.73%** | 크게 개선 |
| 실험5 | dropout=0.05 → 0.10 | 20.54% | 3.78% | 소폭 악화 |
| 실험6 | r=16 → 8 | 23.13% | 4.02% | 악화 |
| 실험7 | epoch=2 → 3 | **13.69%** | **2.61%** | 당시 최고 |

### 최적 하이퍼파라미터

- Base Model: `openai/whisper-large-v3-turbo`
- QLoRA r: **16**
- alpha: **64**
- dropout: **0.05**
- target modules: **q_proj, k_proj, v_proj, out_proj**
- Learning Rate: **1e-4**
- Batch Size: **1**
- Gradient Accumulation: **4**

---

## 2. Epoch 1~5 일반화 / 과적합 비교

> 동일한 최적 설정으로 fresh base에서 5epoch까지 연속 학습  
> fixed100 + 신규500으로 일반화 성능 비교

| Epoch | fixed100 WER | fixed100 CER | 신규500 WER | 신규500 CER | WER Gap | CER Gap | 판단 |
|---:|---:|---:|---:|---:|---:|---:|---|
| 1 | 26.56% | 4.79% | 22.99% | 4.42% | -3.57%p | -0.37%p | 초기 학습 |
| 2 | 19.19% | 3.46% | 16.68% | 3.46% | -2.51%p | 0.00%p | 개선 |
| 3 | 11.93% | 2.42% | 15.06% | 3.20% | +3.13%p | +0.78%p | 개선 지속 |
| **4** | **10.68%** | 2.21% | **13.97%** | **3.05%** | +3.29%p | +0.84%p | ✅ train300 단계 일반화 최고 |
| 5 | 11.00% | **2.18%** | 14.47% | 3.12% | +3.48%p | +0.94%p | WER 소폭 악화 |

**결론:** 현재 실험 범위에서는 **Epoch4가 최적**이었다. Epoch5에서 CER은 미세하게 개선됐지만 fixed100/new500 WER이 모두 악화되어 Epoch4를 선택했다.

> ※ 실험7의 Epoch3 결과(13.69%)와 위 표의 Epoch3 결과(11.93%)는 서로 다른 fresh training run 결과이므로 동일 checkpoint로 취급하지 않는다.

---

## 3. 오류 분석 기반 Train1000 취약유형 보강

신규500 오류 분석 결과 주요 취약유형은 다음과 같았다.

| 유형 | 문장수 | WER | CER |
|---|---:|---:|---:|
| 긴문장 | 76 | **16.61%** | **4.18%** |
| 숫자/단위 | 225 | 14.84% | 3.71% |
| 일반 | 14 | 14.14% | 2.53% |
| 조리동작 | 431 | 13.94% | 3.07% |
| 받침/발음 | 414 | 13.81% | 3.00% |
| 재료명 | 415 | 13.28% | 2.97% |

오류 분석을 바탕으로 전체 레시피 문장 풀에서 취약유형을 우선 보강하여 **train1000**을 구성했다.

### Train300 → Train1000 비교

| 모델 | 학습 데이터 | fixed100 WER | fixed100 CER | 신규500 WER | 신규500 CER |
|---|---|---:|---:|---:|---:|
| train300 Epoch4 | 300개 | 10.68% | 2.21% | 13.97% | 3.05% |
| **train1000 Epoch4** | 취약유형 보강 1000개 (실학습800 / val200) | **7.26%** | **1.49%** | **10.98%** | **2.33%** |

### 숫자/단위 KPI

| 지표 | train300 | train1000 | 변화 |
|---|---:|---:|---:|
| WER | 14.84% | **10.54%** | **-4.30%p** |
| CER | 3.71% | **2.68%** | **-1.03%p** |
| 인식 성공률 | 70.75% | **86.79%** | **+16.04%p** |
| 정확 인식 | 75 / 106 | **92 / 106** | **+17문장** |
| 오인식 | 31 / 106 | **14 / 106** | **-17문장** |

**결론:** 하이퍼파라미터를 고정한 상태에서 학습데이터의 양과 구성을 개선하자 전체 WER/CER와 숫자·단위 인식 성능이 함께 크게 향상되었다.

---

## 4. 숫자/단위 집중 보강 실험

train1000 BEST에서 숫자/단위 취약문장 **250개만 추가 1epoch 학습**했다.

| 모델 | fixed100 WER | fixed100 CER | 신규500 WER | 신규500 CER | 숫자/단위 성공률 | 판단 |
|---|---:|---:|---:|---:|---:|---|
| train1000 BEST | **7.26%** | 1.49% | **10.98%** | 2.33% | 86.79% | 전체 성능 우수 |
| 숫자특화 reinforce250 | 8.20% | 1.54% | 11.07% | **2.32%** | **90.57%** | 숫자 특화 성공, 전체 WER 소폭 악화 |

**결론:** 숫자/단위 성공률은 **86.79% → 90.57%**로 상승했지만 전체 WER은 소폭 악화되었다. 특정 취약유형만 집중 학습하면 해당 유형에는 강해지지만 기존 일반 성능 일부가 희생될 수 있음을 확인했다.

---

## 5. Replay + 숫자보강 혼합학습(MIX750)

특화학습의 장점은 유지하면서 일반 성능 저하를 줄이기 위해 train1000 BEST에서 다음 데이터를 1epoch 추가학습했다.

- 기존 train1000 replay: **500문장**
  - 일반/비숫자 400
  - 숫자/단위 100
- 숫자/단위 보강: **250문장**
- 총 추가학습: **750문장**
- Epoch: **1**
- Learning Rate: **1e-5**

### 최종 모델 비교

| 모델 | fixed100 WER | fixed100 CER | 신규500 WER | 신규500 CER | 숫자/단위 성공률 | 판단 |
|---|---:|---:|---:|---:|---:|---|
| train1000 BEST | **7.26%** | 1.49% | 10.98% | 2.33% | 86.79% | fixed100 WER 절대최저 |
| 숫자특화 reinforce250 | 8.20% | 1.54% | 11.07% | 2.32% | **90.57%** | 숫자/단위 특화 |
| **MIX750** | 7.68% | **1.44%** | **10.72%** | **2.26%** | **90.57%** | ✅ 종합 최종 BEST |

**결론:** MIX750은 train1000 BEST보다 fixed100 WER이 0.42%p 높았지만, 신규500 WER/CER와 fixed100 CER이 더 좋아졌고 숫자/단위 성공률 90.57%를 유지했다. 따라서 일반화 성능과 도메인 핵심정보 인식 성능을 함께 고려한 **종합 최종 BEST**로 선정했다.

---

## 6. 최종 MIX750 종합 성능 평가

> 평가 기준: 신규500  
> WER/CER는 표준 STT 지표이며, 숫자/단위·재료명·조리동작·핵심정보 성공률은 ChefEar 도메인 자체평가 지표이다.

| 평가지표 | 결과 | 해석 |
|---|---:|---|
| 전체 WER | **10.72%** | 전체 단어 오류율 |
| 전체 CER | **2.26%** | 전체 문자 오류율 |
| 문장 Exact Match | **61.80%** | 문장 전체가 완전히 일치한 비율 |
| 숫자/단위 성공률 | **90.57%** | 수량·단위 핵심정보 인식률 |
| 재료명 성공률 | **98.31%** | 등록된 재료명 핵심어 인식률 |
| 조리동작 성공률 | **99.54%** | 조리동작 핵심어 인식률 |
| 긴문장 WER | **11.21%** | 50자 이상 긴문장 단어 오류율 |
| 긴문장 CER | **2.88%** | 50자 이상 긴문장 문자 오류율 |
| 긴문장 완전정답률 | **43.42%** | 긴문장 전체 완전일치 비율 |
| 핵심정보 종합 성공률 | **96.23%** | 숫자/단위·재료명·조리동작 핵심정보 종합 자체평가 |

### 종합 해석

- 전체 신규500 WER/CER는 **10.72% / 2.26%**
- 문장 전체 완전일치율은 **61.80%**
- 숫자/단위는 **90.57%**
- 재료명은 **98.31%**
- 조리동작은 **99.54%**
- 핵심정보 종합 성공률은 **96.23%**
- 긴문장 WER은 **11.21%**로 전체 WER과 큰 차이는 없지만, 긴문장 완전정답률은 **43.42%**로 상대적으로 낮아 문장이 길어질수록 일부 단어 오류가 남는 경향이 확인되었다.

---

## 7. 최종 결론

1. 현재 실험 범위에서 **Epoch4가 가장 안정적인 일반화 성능**을 보였다.
2. QLoRA는 `r=16 / alpha=64 / dropout=0.05 / q,k,v,out / LR=1e-4` 조합이 가장 좋은 결과를 보였다.
3. 하이퍼파라미터 최적화 이후에는 추가 파라미터 변경보다 **학습데이터의 양과 구성 개선이 더 큰 성능 향상**을 만들었다.
4. 오류 분석 기반 train1000 보강으로 신규500 WER이 **13.97% → 10.98%**, 숫자/단위 성공률이 **70.75% → 86.79%**로 향상되었다.
5. 숫자/단위만 집중 보강했을 때 성공률은 **90.57%**까지 상승했지만 전체 WER이 소폭 악화되어 특화학습의 한계가 확인되었다.
6. Replay500 + 숫자보강250을 혼합한 MIX750은 숫자/단위 성공률 **90.57%를 유지하면서 신규500 WER/CER를 10.72% / 2.26%까지 개선**했다.
7. 최종 MIX750은 도메인 핵심정보 종합 성공률 **96.23%**, 재료명 **98.31%**, 조리동작 **99.54%**를 기록해 요리 도메인 핵심정보 인식에서도 높은 성능을 보였다.

### 최종 모델 구분

| 구분 | 모델 | 대표 성능 |
|---|---|---|
| fixed100 WER 절대최저 | train1000 BEST | **7.26%** |
| 숫자/단위 특화 | reinforce250 | **90.57%** |
| **최종 종합 BEST** | **MIX750** | 신규500 WER **10.72%**, 핵심정보 성공률 **96.23%** |

### 최종 BEST 성능

| 평가 지표 | MIX750 |
|---|---:|
| fixed100 WER | 7.68% |
| fixed100 CER | **1.44%** |
| 신규500 WER | **10.72%** |
| 신규500 CER | **2.26%** |
| 숫자/단위 성공률 | **90.57%** |
| 재료명 성공률 | **98.31%** |
| 조리동작 성공률 | **99.54%** |
| 핵심정보 종합 성공률 | **96.23%** |

**최종 모델:** `BEST_FINAL_mix750_replay_numeric`

**최종 판단:**  
하이퍼파라미터 최적화 → Epoch 검증 → 오류 분석 → 취약유형 데이터 보강 → 숫자 특화학습 → 특화학습의 부작용 확인 → Replay 혼합학습 순으로 개선했다. 최종 MIX750은 전체 일반화 성능과 숫자·단위, 재료명, 조리동작 등 ChefEar 핵심정보 인식 성능 사이에서 가장 균형 잡힌 결과를 보여 **최종 종합 BEST**로 선정했다.